# 069 — The Attention Mechanism

Lesson 068 ended with a fixed context vector that had to hold everything.
Attention deletes it: keep **every** encoder state and let the decoder build a
different context at each output step. The budget per input token stops
shrinking — **20 numbers per token at length 100 becomes a constant 1,000**.

Part D trains a real attention layer by hand; it takes a couple of minutes.

| Part | What we check |
|---|---|
| A | the fixed vector against attention, as arithmetic |
| B | one decoder step, computed |
| C | what it costs: scores are $T_{in} \times T_{out}$ |
| D | can it learn an alignment nobody supplied? |
| E | the control that learns nothing — and why |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — The budget per token

In [ ]:
H = 1000
print(f"  {'length':>8}{'fixed vector':>15}{'per token':>11}"
      f"{'attention sees':>17}{'per token':>11}")
for length in (5, 20, 50, 100):
    print(f"  {length:>8}{2 * H:>15,}{2 * H / length:>11.0f}"
          f"{length * H:>17,}{H:>11,}")

assert 2 * H / 100 == 20          # the 2014 bottleneck at length 100
assert H == 1000                  # attention's budget, at every length
print()
print("The fourth column grows with the sentence and the fifth does not move.")
print("That is the whole idea: nothing has to be squeezed out to make room.")

## Part B — One decoder step

Score every encoder state against the decoder's state, softmax the scores into
weights, and take the weighted sum. Three lines.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
T, H_small = 6, 8

def softmax(z, axis=-1):
    z = z - z.max(axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis, keepdims=True)

encoder_states = rng.normal(0, 1, (T, H_small))     # kept, not summarised
decoder_state = rng.normal(0, 1, H_small)
Wa = rng.normal(0, 0.3, (H_small, H_small))

scores = (decoder_state @ Wa) @ encoder_states.T    # 1. one score per input
weights = softmax(scores)                           # 2. they sum to 1
context = weights @ encoder_states                  # 3. a weighted sum

print("scores :", scores.round(2))
print("weights:", weights.round(3), "sum", weights.sum().round(6))
print("context shape:", context.shape, "- the same width as one state")
assert abs(weights.sum() - 1) < 1e-9
assert context.shape == (H_small,)
print()
print("A different decoder state gives different weights, so the context is")
print("rebuilt at every output step. There is no single summary any more.")
other = softmax((rng.normal(0, 1, H_small) @ Wa) @ encoder_states.T)
assert not np.allclose(weights, other)
print("two decoder states -> peaks at inputs", weights.argmax(), "and", other.argmax())

## Part C — What it costs

In [ ]:
for length in (10, 50, 100, 1000):
    print(f"  {length:>5} tokens -> {length * length:>9,} alignment scores")
assert 100 * 100 == 10_000 and 1000 * 1000 == 1_000_000
print()
print("Quadratic in the length. Comfortable for a sentence, and the binding")
print("constraint for a document - which is where the transformer story goes.")

## Part D — Can it learn an alignment nobody supplied?

Task: reverse a sequence of six tokens. The correct alignment is the
anti-diagonal, and **nothing in the loss mentions alignment** — only the output
tokens. Baseline for "the weight peaks in the right place" is 1/6 = 17%.

This trains for a couple of minutes.

In [ ]:
V, H = 10, 32
rng = np.random.default_rng(1)

E = rng.normal(0, 0.5, (V, H))
Wx, Wh = rng.normal(0, 0.3, (H, H)), rng.normal(0, 0.3, (H, H))
Wa, Wo = rng.normal(0, 0.1, (H, H)), rng.normal(0, 0.1, (H, V))
S = rng.normal(0, 0.5, (T, H))          # one learned query per output step

def encode(x, recurrent=True):
    if not recurrent:
        return E[x]                     # content only - no position anywhere
    states, h = [], np.zeros(H)
    for t in range(T):
        h = np.tanh(E[x[t]] @ Wx + h @ Wh)
        states.append(h)
    return np.array(states)

def forward(x, recurrent=True):
    states = encode(x, recurrent)
    A = softmax((S @ Wa) @ states.T)
    return states, A, softmax((A @ states) @ Wo)

def evaluate(n=200, recurrent=True):
    tokens = aligned = 0.0
    for _ in range(n):
        x = rng.integers(0, V, T)
        _, A, P = forward(x, recurrent)
        tokens += (P.argmax(1) == x[::-1]).mean()
        aligned += np.mean([np.argmax(r) == T - 1 - i for i, r in enumerate(A)])
    return tokens / n, aligned / n

def train(steps, lr=0.5, recurrent=True, every=1000):
    global Wa, Wo, S
    print(f"  {'step':>6}{'loss':>9}{'token acc':>12}{'alignment':>12}")
    for step in range(steps + 1):
        X = rng.integers(0, V, (24, T))
        gWa, gWo, gS, total = (np.zeros_like(Wa), np.zeros_like(Wo),
                               np.zeros_like(S), 0.0)
        for x in X:
            y = x[::-1]
            states = encode(x, recurrent)
            A = softmax((S @ Wa) @ states.T)
            C = A @ states
            P = softmax(C @ Wo)
            total += -np.log(P[np.arange(T), y] + 1e-12).mean()
            dP = P.copy()
            dP[np.arange(T), y] -= 1
            dP /= T
            gWo += C.T @ dP
            dA = (dP @ Wo.T) @ states.T
            ds = A * (dA - (dA * A).sum(1, keepdims=True))
            gWa += S.T @ (ds @ states)
            gS += (ds @ states) @ Wa.T
        Wo -= lr * gWo / len(X)
        Wa -= lr * gWa / len(X)
        S -= lr * gS / len(X)
        if step % every == 0:
            acc, align = evaluate(200, recurrent)
            print(f"  {step:>6}{total / len(X):>9.3f}{acc:>11.1%}{align:>12.0%}")
    return evaluate(300, recurrent)

print("baseline for a correct alignment peak: 1/6 =", f"{1/6:.0%}")
acc, align = train(3000)
print(f"\nafter 3,000 steps: token accuracy {acc:.1%}, alignment {align:.0%}")
assert align > 1 / 6
print("The lesson's full 8,000-step run reaches about 47% alignment; the point")
print("here is that it beats the 17% baseline at all, having never been told")
print("what an alignment is.")

In [ ]:
x = rng.integers(0, V, T)
_, A, _ = forward(x)
print("learned alignment (row = output step, col = input position):")
for i, row in enumerate(A):
    hit = "correct" if np.argmax(row) == T - 1 - i else ""
    print(f"  out {i}: " + " ".join(f"{v:.2f}" for v in row)
          + f"   peak {np.argmax(row)}  want {T - 1 - i}  {hit}")
print()
print("Partial, and the pattern is explainable: late output steps attend to")
print("EARLY input positions, whose recurrent states are clean, while early")
print("outputs need LATE states that blend everything before them.")

## Part E — The control that learns nothing

Replace the recurrent encoder with a plain embedding lookup, so a state carries
**content and no position**. The alignment task should now be impossible.

In [ ]:
# fresh parameters, same task, content-only states
Wa = rng.normal(0, 0.1, (H, H))
Wo = rng.normal(0, 0.1, (H, V))
S = rng.normal(0, 0.5, (T, H))

acc_flat, align_flat = train(2000, recurrent=False)
print(f"\ncontent-only states: token accuracy {acc_flat:.1%}, "
      f"alignment {align_flat:.0%}")
assert abs(align_flat - 1 / 6) < 0.06        # flat, at the baseline
print()
print("Flat at the 17% baseline: two identical tokens at different positions")
print("have identical states, so there is no positional signal for alignment")
print("to find. Content-only representations cannot express positional")
print("alignment - which is exactly the gap positional encoding fills in a")
print("transformer (lesson 074).")

## What this notebook established

- A fixed context gives the decoder **2H** numbers whatever the length — 20 per
  token at length 100 — while attention gives it **T·H**, so the per-token
  budget is a constant **1,000**.
- One decoder step is three lines: score, softmax, weighted sum. A different
  decoder state produces a different context, so there is no single summary.
- Alignment scores are $T_{in} \times T_{out}$ — **10,000** at length 100 and a
  **million** at 1,000.
- Trained only on the output tokens, attention learned an alignment it was
  never shown, beating the 17% baseline.
- With **content-only** encoder states the alignment stayed flat at the
  baseline: identical tokens at different positions are indistinguishable. That
  is the measured reason transformers need positional encoding.

## Exercises

1. Train for 8,000 steps and compare with the lesson's ~47%.
2. Add a position index to the content-only states and re-run Part E. How much
   of the alignment comes back?
3. Make the task "copy the sequence" instead. Does the alignment become easier
   to learn, and why?